In [13]:
# 1. Importing libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error
import pickle
import os


1️⃣ Data Collection

In [14]:
def load_ratings(path='ml-100k/u.data'):
    r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
    ratings = pd.read_csv(path, sep='\t', names=r_cols, encoding='latin-1')
    print("Ratings data shape:", ratings.shape)
    return ratings

2️⃣ Data Preprocessing, Training & Test data

In [15]:

def compute_user_similarity(data_matrix):
    """
    Computes the cosine similarity between users based on their movie ratings.

    Cosine similarity measures how similar two users are by comparing the direction of their rating vectors,
    regardless of their magnitude. The result is a matrix where each entry [i, j] represents the similarity
    between user i and user j:
      - 1 means very similar tastes,
      - 0 means no similarity,
      - -1 means opposite tastes.

    The diagonal is set to 0 to ignore self-similarity.

    Parameters:
        data_matrix (pd.DataFrame): User-movie ratings matrix (users as rows, movies as columns).

    Returns:
        np.ndarray: User-user similarity matrix.
    """
    user_similarity = cosine_similarity(data_matrix)
    np.fill_diagonal(user_similarity, 0)
    return user_similarity

In [16]:
def train_test_split(df, test_size=0.1, random_state=None):
    """
    Splits a pandas DataFrame into training and testing sets.

    Parameters:
        df (pd.DataFrame): The DataFrame to split.
        test_size (float): Fraction of the data to use as the test set (default is 0.1).
        random_state (int, optional): Seed for reproducibility.

    Returns:
        train (pd.DataFrame): The training set.
        test (pd.DataFrame): The test set.
    """
    test = df.sample(frac=test_size, random_state=random_state)
    train = df.drop(test.index).reset_index(drop=True)
    test = test.reset_index(drop=True)
    return train, test

3️⃣ Model Building (Matrix creation) and saving the model

In [17]:
def compute_user_similarity(data_matrix):
    """
    Computes the cosine similarity between users based on their movie ratings.

    Cosine similarity measures how similar two users are by comparing the direction of their rating vectors,
    regardless of their magnitude. The result is a matrix where each entry [i, j] represents the similarity
    between user i and user j:
      - 1 means very similar tastes,
      - 0 means no similarity,
      - -1 means opposite tastes.

    The diagonal is set to 0 to ignore self-similarity.

    Parameters:
        data_matrix (pd.DataFrame): User-movie ratings matrix (users as rows, movies as columns).

    Returns:
        np.ndarray: User-user similarity matrix.
    """
    user_similarity = cosine_similarity(data_matrix)
    np.fill_diagonal(user_similarity, 0)
    return user_similarity

In [18]:
def create_user_movie_matrix(ratings):
    """
    Creates a user-movie ratings matrix from the ratings DataFrame.
    Rows: user_id, Columns: movie_id, Values: rating (0 if not rated)
    """
    return ratings.pivot_table(index='user_id', columns='movie_id', values='rating').fillna(0)

4️⃣ Recommendation Function

In [19]:
def recommend_movies_user_based(user_id, data_matrix, user_similarity, n_recommendations=5):
    """
    Recommends movies to a user based on user-based collaborative filtering.

    This function predicts ratings for all movies the user hasn't rated yet, using the ratings of similar users.
    It then recommends the top N movies with the highest predicted ratings.

    Parameters:
        user_id (int): The ID of the user to recommend movies for.
        data_matrix (pd.DataFrame): User-movie ratings matrix (users as rows, movies as columns).
        user_similarity (np.ndarray): User-user similarity matrix.
        n_recommendations (int): Number of movies to recommend (default is 5).

    Returns:
        list: List of recommended movie IDs.
    """
    user_idx = user_id - 1  # user_id starts from 1
    sim_scores = user_similarity[user_idx]
    weighted_ratings = np.dot(sim_scores, data_matrix)
    sum_sim = np.array([np.abs(sim_scores).sum()] * data_matrix.shape[1])
    predicted_ratings = weighted_ratings / np.where(sum_sim == 0, 1, sum_sim)
    already_rated = data_matrix.iloc[user_idx].to_numpy().nonzero()[0]
    predicted_ratings[already_rated] = -np.inf
    recommended_movie_indices = np.argsort(predicted_ratings)[::-1][:n_recommendations]
    recommended_movie_ids = data_matrix.columns[recommended_movie_indices]
    return recommended_movie_ids.tolist()

5️⃣ Evaluation Function (RMSE)

In [20]:
def predict_rating(user_id, movie_id, data_matrix, user_similarity):
    """
    Predicts the rating a user would give to a specific movie using user-based collaborative filtering.

    The prediction is made by taking a weighted average of ratings from other users who have rated the movie,
    where the weights are the similarity scores between the target user and other users.

    Parameters:
        user_id (int): The ID of the user for whom to predict the rating.
        movie_id (int): The ID of the movie to predict the rating for.
        data_matrix (pd.DataFrame): User-movie ratings matrix (users as rows, movies as columns).
        user_similarity (np.ndarray): User-user similarity matrix.

    Returns:
        float: The predicted rating, or np.nan if prediction is not possible.
    """
    user_idx = user_id - 1
    if movie_id not in data_matrix.columns:
        return np.nan
    sim_scores = user_similarity[user_idx]
    movie_ratings = data_matrix[movie_id]
    mask = movie_ratings > 0
    if mask.sum() == 0:
        return np.nan
    pred = np.dot(sim_scores[mask], movie_ratings[mask]) / np.abs(sim_scores[mask]).sum()
    return pred

In [21]:
def evaluate_rmse(test, data_matrix, user_similarity):
    y_true, y_pred = [], []
    for _, row in test.iterrows():
        pred = predict_rating(row['user_id'], row['movie_id'], data_matrix, user_similarity)
        if not np.isnan(pred):
            y_true.append(row['rating'])
            y_pred.append(pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"User-based CF RMSE on test set: {rmse:.4f}")
    print(f"Rating scale: 1 to 5")
    if rmse < 1:
        print("Excellent: Predictions are very close to actual ratings.")
    elif rmse < 1.5:
        print("Good: Predictions are reasonably close to actual ratings.")
    else:
        print("Needs improvement: Predictions are not very close to actual ratings.")
    return rmse

6️⃣ Main Workflow


In [22]:
def save_model(model, filename):
    """
    Saves the given model to a file using pickle.

    Parameters:
        model (dict): The model to save (e.g., train_matrix, user_similarity).
        filename (str): The name of the file to save the model to.
    """
    with open(filename, "wb") as f:
        pickle.dump(model, f)
    print(f"Model saved to {filename}")

In [23]:
def load_model(filename):
    with open(filename, "rb") as f:
        model = pickle.load(f)
    print(f"Model loaded from {filename}")
    return model

In [24]:
if __name__ == "__main__":
    # Load data
    ratings = load_ratings()

    # Split data for evaluation
    train, test = train_test_split(ratings, test_size=0.1, random_state=42)

    # Build train matrix and similarity
    train_matrix = create_user_movie_matrix(train)
    user_similarity = compute_user_similarity(train_matrix)

    # Recommend movies for a user
    recommended = recommend_movies_user_based(1, train_matrix, user_similarity, n_recommendations=5)
    print("User-based CF - Recommended movie IDs for user 1:", recommended)

    # Evaluate model
    evaluate_rmse(test, train_matrix, user_similarity)
    
    # Create the model dictionary
    model = {
    "train_matrix": train_matrix,
    "user_similarity": user_similarity
    }
    
    # Save the model to a file
    save_model(model, "ml_models/user_cf_model.pkl")
    
    # Load the saved model
    loaded_model = load_model("ml_models/user_cf_model.pkl")
    print(loaded_model.keys())  # Should print: dict_keys(['train_matrix', 'user_similarity'])

Ratings data shape: (100000, 4)
User-based CF - Recommended movie IDs for user 1: [181, 100, 318, 258, 423]
User-based CF - Recommended movie IDs for user 1: [181, 100, 318, 258, 423]
User-based CF RMSE on test set: 1.0120
Rating scale: 1 to 5
Good: Predictions are reasonably close to actual ratings.
Model saved to ml_models/user_cf_model.pkl
Model loaded from ml_models/user_cf_model.pkl
dict_keys(['train_matrix', 'user_similarity'])
User-based CF RMSE on test set: 1.0120
Rating scale: 1 to 5
Good: Predictions are reasonably close to actual ratings.
Model saved to ml_models/user_cf_model.pkl
Model loaded from ml_models/user_cf_model.pkl
dict_keys(['train_matrix', 'user_similarity'])
